In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\pramo\AppData\Local\Temp\ipykernel_22664\2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,doc_content_chars_max=200
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
web_doc = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 250
)
chunks = text_splitter.split_documents(web_doc)
# embeddings = OllamaEmbeddings(model="nano")
vector_store = FAISS.from_documents(chunks, OllamaEmbeddings(model="nomic-embed-text"))
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000189EDBDF380>, search_kwargs={'k': 3})

In [ ]:
from langchain_core.tools import create_retriever_tool

langsmith_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "search for information about langsmith. For any question about langsmith, you must use this tool"
)

In [8]:
# from langchain_community.utilities import ArxivAPIWrapper
# from langchain_community.tools import ArxivQueryRun

# arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_search_max = 200)
# arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
from langchain.tools import tool
import arxiv

client = arxiv.Client()

@tool
def search_arxiv(query: str) -> str:
    """Search Arxiv for research papers."""
    search = arxiv.Search(query=query, max_results=3)

    papers = []
    for paper in client.results(search):
        papers.append(
            f"Title: {paper.title}\n"
            f"Summary: {paper.summary[:300]}"
        )

    return "\n\n".join(papers)

In [10]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print(type(llm))

<class 'langchain_groq.chat_models.ChatGroq'>


In [ ]:
tools=[
    wiki_tool,
    langsmith_tool,
    search_arxiv
]

binding_tools = llm.bind_tools(tools)



In [16]:
##pulling promts from langchain hub
from langsmith import Client

langsmith_client = Client()

prompt = langsmith_client.pull_prompt(
    "hwchase17/openai-functions-agent",
    dangerously_pull_public_prompt=True
)

In [ ]:
##creating agents 
from langchain.agents import create_agent
system_prompt = prompt.messages[0].prompt.template
agent = create_agent(
    model=llm,
    tools=binding_tools,
    system_prompt=system_prompt
)
# tools[0].invoke("Python")
# tools[1].invoke("What is LangSmith?")
# tools[2].invoke("Attention Is All You Need")

In [ ]:
res = agent.invoke(
    {
        "messages": [
               { "role":"system",
                "content":"You are a helpful assistant."
               },{
                   
                "role": "user",
                "content": "whats the 1605.08386 paper about?"
               }
        ]
    }
)
print(res)

In [ ]:
##agent executor 
from langchain_classic.agents import AgentExecutor
agent_exe = AgentExecutor(agent = agent,tools=tools,verbose=True)
agent_exe

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=<langgraph.graph.state.CompiledStateGraph object at 0x0000023F7665C410>, input_keys_arg=[], return_keys_arg=[], stream_runnable=True), tools=[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\pramo\\OneDrive\\Desktop\\ChatBot\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)), StructuredTool(name='langsmith_search', description='search for information about langsmith. For any question about langsmith, you must use this tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000023F73A079C0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000023F73A04360>), ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.Unexpecte